In [19]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
from statsmodels.discrete.discrete_model import MNLogit


In [9]:
# CSV読み込み
df = pd.read_csv("/Users/furuken/Documents/Kyushu_University/master's_thesis/data/processed/df_clustered_with_features.csv")

# 並び替え（重要）
df = df.sort_values(["コード", "年度"]).reset_index(drop=True)

# 列名一覧
df.columns


Index(['コード', '年度', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8',
       'PC9', 'PC10', 'PC11', 'Cluster', '総資産_log', '売上高_log', '短期借入金_log',
       '長期借入金_log', '利益剰余金_log', '現金同等物_log', '営業利益_log', '営業CF_log',
       '投資CF_log', '財務CF_log', '自己資本比率', '銘柄名'],
      dtype='object')

In [11]:
df["Cluster_t1"] = df.groupby("コード")["Cluster"].shift(-1)

df[["コード", "年度", "Cluster", "Cluster_t1"]].head(10)


,コード,年度,Cluster,Cluster_t1
0,1301,2010,3,4.0
1,1301,2011,4,4.0
2,1301,2012,4,4.0
3,1301,2013,4,4.0
4,1301,2014,4,4.0
5,1301,2015,4,3.0
6,1301,2016,3,3.0
7,1301,2017,3,3.0
8,1301,2018,3,3.0
9,1301,2019,3,3.0


In [12]:
df["cluster_transition"] = (
    (df["Cluster"] != df["Cluster_t1"]) &
    (df["Cluster_t1"].notna())
).astype(int)


In [13]:
df_model = df[df["Cluster_t1"].notna()].copy()


In [14]:
y = df_model["cluster_transition"]

y.value_counts(normalize=True)


cluster_transition
0    0.641005
1    0.358995
Name: proportion, dtype: float64

In [ ]:
X_pc = df_model[["PC1", "PC2", "PC3"]]

In [ ]:
cluster_dummies = pd.get_dummies(
    df_model["Cluster"],
    prefix="cluster",
    drop_first=True
)

In [21]:
# クラスタダミーを int に変換
cluster_dummies = cluster_dummies.astype(int)

# X 再構築
X = pd.concat([X_pc, cluster_dummies], axis=1)
X = sm.add_constant(X)

# 確認
X.dtypes


const        float64
PC1          float64
PC2          float64
PC3          float64
cluster_1      int64
cluster_2      int64
cluster_3      int64
cluster_4      int64
dtype: object

In [22]:
X = pd.concat([X_pc, cluster_dummies], axis=1)
X = sm.add_constant(X)

X.head()

,const,PC1,PC2,PC3,cluster_1,cluster_2,cluster_3,cluster_4
0,1.0,0.349607,-1.575416,-0.921573,0,0,1,0
1,1.0,0.132577,-3.468161,-0.294505,0,0,0,1
2,1.0,0.259218,-3.259065,-0.048364,0,0,0,1
3,1.0,0.879700,-0.609680,0.400468,0,0,0,1
4,1.0,0.878863,-0.643339,0.280736,0,0,0,1


In [23]:
probit_model = sm.Probit(y, X)
probit_result = probit_model.fit()

print(probit_result.summary())

Optimization terminated successfully.
         Current function value: 0.596819
         Iterations 6
                          Probit Regression Results                           
Dep. Variable:     cluster_transition   No. Observations:                21900
Model:                         Probit   Df Residuals:                    21892
Method:                           MLE   Df Model:                            7
Date:                Tue, 03 Feb 2026   Pseudo R-squ.:                 0.08581
Time:                        11:34:34   Log-Likelihood:                -13070.
converged:                       True   LL-Null:                       -14297.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.7548      0.060     12.621      0.000       0.638       0.872
PC1            0.0197      0.

In [24]:
# 限界効果（平均限界効果）
mfx = probit_result.get_margeff(at="mean")

print(mfx.summary())


       Probit Marginal Effects       
Dep. Variable:     cluster_transition
Method:                          dydx
At:                              mean
                dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
PC1            0.0072      0.002      3.061      0.002       0.003       0.012
PC2           -0.0105      0.003     -3.727      0.000      -0.016      -0.005
PC3           -0.0054      0.006     -0.861      0.389      -0.018       0.007
cluster_1     -0.7766      0.027    -28.292      0.000      -0.830      -0.723
cluster_2     -0.3950      0.024    -16.408      0.000      -0.442      -0.348
cluster_3     -0.3885      0.025    -15.802      0.000      -0.437      -0.340
cluster_4     -0.3894      0.024    -15.983      0.000      -0.437      -0.342


モデル2

In [25]:
# PC（構造）
X_pc = df_model[["PC1", "PC2", "PC3"]]

# 財務余力（水準）
X_fin = df_model[["営業CF_log", "自己資本比率"]]


In [27]:
X = pd.concat(
    [
        X_pc,
        X_fin,
        cluster_dummies
    ],
    axis=1
)


In [28]:
X = sm.add_constant(X)

X.dtypes

const        float64
PC1          float64
PC2          float64
PC3          float64
営業CF_log     float64
自己資本比率       float64
cluster_1      int64
cluster_2      int64
cluster_3      int64
cluster_4      int64
dtype: object

In [29]:
probit_model_2 = sm.Probit(y, X)
probit_result_2 = probit_model_2.fit()

print(probit_result_2.summary())


Optimization terminated successfully.
         Current function value: 0.596515
         Iterations 6
                          Probit Regression Results                           
Dep. Variable:     cluster_transition   No. Observations:                21900
Model:                         Probit   Df Residuals:                    21890
Method:                           MLE   Df Model:                            9
Date:                Tue, 03 Feb 2026   Pseudo R-squ.:                 0.08627
Time:                        13:11:49   Log-Likelihood:                -13064.
converged:                       True   LL-Null:                       -14297.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.0277      0.133      7.755      0.000       0.768       1.287
PC1            0.0251      0.

In [30]:
mfx_2 = probit_result_2.get_margeff(at="mean")
print(mfx_2.summary())


       Probit Marginal Effects       
Dep. Variable:     cluster_transition
Method:                          dydx
At:                              mean
                dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
PC1            0.0092      0.002      3.799      0.000       0.004       0.014
PC2            0.0080      0.008      1.034      0.301      -0.007       0.023
PC3           -0.0192      0.009     -2.232      0.026      -0.036      -0.002
営業CF_log      -0.0016      0.000     -3.506      0.000      -0.002      -0.001
自己資本比率        -0.0009      0.000     -1.767      0.077      -0.002    9.63e-05
cluster_1     -0.8093      0.032    -25.513      0.000      -0.872      -0.747
cluster_2     -0.4312      0.029    -14.831      0.000      -0.488      -0.374
cluster_3     -0.4250      0.030    -14.350      0.000      -0.483      -0.367
cluster_4     -0.4235      0.029    -14.522      0.000    

多項プロビット

In [ ]:
# 遷移した企業だけ抽出
df_mn = df_model[df_model["cluster_transition"] == 1].copy()

# 被説明変数：遷移先クラスタ
y_mn = df_mn["Cluster_t1"]
y_mn


In [34]:
X_mn = df_mn[
    [
        "PC1",
        "PC2",
        "PC3",
        "営業CF_log",
        "自己資本比率"
    ]
]

X_mn = sm.add_constant(X_mn)


In [35]:
mnlogit_model = sm.MNLogit(y_mn, X_mn)
mnlogit_result = mnlogit_model.fit()

print(mnlogit_result.summary())


Optimization terminated successfully.
         Current function value: 1.309257
         Iterations 9
                          MNLogit Regression Results                          
Dep. Variable:             Cluster_t1   No. Observations:                 7862
Model:                        MNLogit   Df Residuals:                     7838
Method:                           MLE   Df Model:                           20
Date:                Tue, 03 Feb 2026   Pseudo R-squ.:                  0.1360
Time:                        13:48:02   Log-Likelihood:                -10293.
converged:                       True   LL-Null:                       -11913.
Covariance Type:            nonrobust   LLR p-value:                     0.000
Cluster_t1=1       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -5.5002      0.495    -11.117      0.000      -6.470      -4.531
PC1              2.2353

初期クラスタ別の分岐

In [36]:
df_branch = df_model[df_model["cluster_transition"] == 1].copy()

In [37]:
route_map = {
    1: "成長・成功",
    2: "成長志向",
    3: "安定化",
    4: "悪化・停滞"
}

df_branch["route"] = df_branch["Cluster_t1"].map(route_map)

In [ ]:
route_share = (
    df_branch
    .pivot_table(
        index="Cluster",
        columns="route",
        values="コード",   
        aggfunc="count",
        fill_value=0
    )
)

# 行ごとに構成比に変換
route_share = route_share.div(route_share.sum(axis=1), axis=0)

route_share


route,安定化,悪化・停滞,成長・成功,成長志向
Cluster,,,,
0,0.274571,0.329173,0.028081,0.368175
1,0.768769,0.141141,0.000000,0.090090
2,0.463339,0.515712,0.020949,0.000000
3,0.000000,0.284033,0.170420,0.545548
4,0.541696,0.000000,0.067274,0.391030


In [41]:
vars_compare = [
    "PC1", "PC2", "PC3",
    "営業CF_log", "自己資本比率"
]

mean_table = (
    df_branch
    .groupby(["Cluster", "route"])[vars_compare]
    .mean()
    .round(3)
)

mean_table


PC1    PC2    PC3  営業CF_log  自己資本比率
Cluster route                                       
0       安定化   -0.276 -0.164  2.971    11.762  42.433
        悪化・停滞 -2.254 -0.249  2.777    11.770  40.685
        成長・成功  1.877 -0.365  3.447    11.853  36.792
        成長志向  -1.834  0.404  2.672    13.689  50.983
1       安定化    2.531  0.039 -0.148    21.426  44.561
        悪化・停滞  2.461 -0.158 -0.046    22.629  36.260
        成長志向   2.206  0.381 -0.247    23.793  50.733
2       安定化   -0.310  0.586 -0.232    20.448  50.901
        悪化・停滞 -2.107  0.331 -0.280    18.443  47.868
        成長・成功  1.750  1.027 -0.253    23.686  55.300
3       悪化・停滞  0.334  0.041 -0.177    20.036  40.455
        成長・成功  2.309 -0.014 -0.212    19.742  45.230
        成長志向   0.157  0.214 -0.352    17.898  50.762
4       安定化   -0.174 -0.811 -0.362    12.370  38.056
        成長・成功  2.115 -0.551 -0.217    16.677  37.879
        成長志向  -1.908 -0.721 -0.530    11.462  42.558